# Developing ML Classification Framework: Severely Imbalanced Biotic-Abiotic Sample Set

##### Introduction:
This notebook builds on the [previous notebook](03_balanced_ml_classification.ipynb) in developing a biotic-abiotic classification framework, whereby four classifiers were trained to identify whether pyrolysis-Gas Chromatography-Mass Spectrometry (py-GC-MS) samples are biotic or abiotic based on their extracted mass spectra. Unlike the previous notebook that focuses on a less imbalanced sample subset, this notebook aims to determine whether geochemical patterns from py-GC-MS data can be used to distinguish between biologically-generated material and non-biological analogues under severe class imbalance.

The same preprocessing, feature engineering, training, hyperparameter tuning, and model evaluation framework is applied in this notebook, but is extended across the entire dataset of 140 biotic and 14 abiotic samples. A detailed breakdown of the full sample set is provided in the following [documentation](../docs/total-sample-breakdown.md). Note that the complete exploration and accompanying markdown documentation for methodological choices, experimentation, and analyses regarding the resulting ML framework can be found in the previous notebook.

##### Imports:

In [1]:
# mlbiosig imports:
from mlbiosig import samples_labels, build_features, engineer_features

# Model imports:
from sklearn.linear_model import LogisticRegression  # LR
from sklearn.ensemble import RandomForestClassifier  # RF
from sklearn.svm import SVC  # SVC
from xgboost import XGBClassifier  # XGB

# ML imports:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    RocCurveDisplay,
    ConfusionMatrixDisplay,
)

# Misc. imports:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import randint
import warnings

### Highly Imbalanced Biotic-Abiotic Sample Set:

##### Generating py-MS Feature Matrix

In [19]:
# Obtain biotic & abiotic py-GC-MS mass spectra CSV filepaths:
filepaths, labels = samples_labels()  # filepaths and label ('biotic'/'abiotic')
print(
    f"Total Samples: {len(filepaths)} | Total Biotic: {labels.count('biotic')} | Total Abiotic: {labels.count('abiotic')}"
)

Total Samples: 154 | Total Biotic: 140 | Total Abiotic: 14


In [3]:
# Build feature matrix:
feature_matrix, flags = build_features(filepaths, labels)

feature_matrix.head(5)  # first 5 samples

Preprocessing py-GC-MS samples:   0%|          | 0/154 [00:00<?, ?sample/s]

m_z,50.1,50.3,50.8,51.0,51.1,51.3,52.0,52.1,52.2,52.3,...,21.6,27.3,31.3,16.3,14.8,40.3,40.0,14.1,32.2,44.3
041121_SP_0,0.196884,0.000166,0.000001,0.000445,0.350330,0.000039,0.000741,0.155981,0.155879,0.000102,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
041121_SP_3H_E,0.178774,0.000005,0.000000,0.000044,0.328891,0.000001,0.000047,0.000000,0.294349,0.000008,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
041122_SP_3H_C,0.188820,0.000023,0.000000,0.000012,0.286135,0.000000,0.000111,0.150565,0.158116,0.000014,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
080206GRSSPLT,0.015949,0.000000,0.000000,0.000000,0.080974,0.000000,0.000000,0.040579,0.000132,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
081221_CH_24H,0.167491,0.000020,0.000000,0.000136,0.307936,0.000004,0.000050,0.228654,0.040986,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


##### Performing Feature Engineering

In [18]:
# Separate features & target, split test/train, engineer features ready for ML:
X_train, X_test, y_train, y_test, le, selector = engineer_features(feature_matrix)
print(f"Training set: {len(X_train)} | Testing set: {len(X_test)}")

Training set: 123 | Testing set: 31


In [5]:
# Stratified CV folds due to (a) imbalanced and (b) small dataset:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

##### ML Biotic-Abiotic Classification: Overview

As documented and explained in the previous notebook, the four classifiers trained and evaluated in this work are Logistic Regression (LR), Support Vector Classifier (SVC), Random Forest (RF) Classifier, and XGBoost.
##### Handling Biotic-Abiotic Imbalance

In [6]:
# Calculating class imbalance ratio for XGBoost:
abiotic_samples = (y_train == 0).sum()  # abiotic encoded as 0
biotic_samples = (y_train == 1).sum()  # biotic encoded as 1

imbalance_ratio = abiotic_samples / biotic_samples
print(f"Class imbalance ratio (scale_pos_weight): {imbalance_ratio:.3f}")

Class imbalance ratio (scale_pos_weight): 0.098


##### Building ML Pipelines

In [7]:
# Logistic Regression:
lr_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                random_state=42,
                max_iter=2000,  # high for convergence warning
            ),
        ),
    ]
)

# Support Vector Classifier (SVC):
svc_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "classifier",
            SVC(
                class_weight="balanced",
                random_state=42,
                max_iter=2000,  # high for convergence warning
                probability=True,
            ),
        ),
    ]
)

# Random Forest:
rf_pipeline = Pipeline(
    [
        ("scaler", RobustScaler()),
        (
            "classifier",
            RandomForestClassifier(class_weight="balanced", random_state=42),
        ),
    ]
)

# XGBoost:
xgb_pipeline = Pipeline(
    [
        ("scaler", RobustScaler()),
        (
            "classifier",
            XGBClassifier(
                scale_pos_weight=imbalance_ratio,
                random_state=42,
                eval_metric="logloss",  # binary cross entropy for comparisons
            ),
        ),
    ]
)

##### Hyperparameter Tuning

Before comparing the performance of the four models, the hyperparameter combinations of each classifier is optimised to best maximise its performance. A coarse-to-fine hyperparameter tuning strategy is adopted with fixed randomised search ranges for each classifier, as comprehensively documented in the previous notebook.

In [8]:
# Hide FutureWarning for better readability:
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn.svm")

In [9]:
# Coarse Tuning via RandomSearch:
lr_random_param = {
    "classifier__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "classifier__solver": ["lbfgs", "liblinear"],
}

# Perform search and fit on training data:
lr_random_search = RandomizedSearchCV(
    lr_pipeline, lr_random_param, cv=cv, scoring="f1", random_state=42  # F1 scoring
)
lr_random_search.fit(X_train, y_train)

# Best parameter combination & F1 score:
print(f"LR Random Search Best Params: {lr_random_search.best_params_}")  # best params
print(f"LR Random Search Best CV F1: {lr_random_search.best_score_:.3f}")  # best F1

best_C_lr = lr_random_search.best_params_["classifier__C"]
best_solver_lr = lr_random_search.best_params_["classifier__solver"]

LR Random Search Best Params: {'classifier__solver': 'lbfgs', 'classifier__C': 100}
LR Random Search Best CV F1: 0.973


In [10]:
# Fine Tuning via GridSearch: Dependent on random search space
lr_grid_param = {
    "classifier__C": [
        best_C_lr * 0.1,
        best_C_lr * 0.5,
        best_C_lr,  # based on random search
        best_C_lr * 2,
        best_C_lr * 10,
    ],
    "classifier__solver": [best_solver_lr],
}

# Perform search and fit on training data:
lr_grid_search = GridSearchCV(lr_pipeline, lr_grid_param, cv=cv, scoring="f1")
lr_grid_search.fit(X_train, y_train)

# Best parameter combination & F1 score:
print(f"LR Fine-Tuned Grid Search Best Params: {lr_grid_search.best_params_}")
print(f"LR Fine-Tuned Search Best CV F1: {lr_grid_search.best_score_:.3f}")  # best F1

LR Fine-Tuned Grid Search Best Params: {'classifier__C': 100, 'classifier__solver': 'lbfgs'}
LR Fine-Tuned Search Best CV F1: 0.973


In [11]:
# Coarse Tuning via RandomSearch:
svc_random_param = {
    "classifier__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "classifier__kernel": ["linear", "rbf"],
    "classifier__gamma": ["scale", "auto"],
}

# Perform search and fit on training data:
svc_random_search = RandomizedSearchCV(
    svc_pipeline, svc_random_param, cv=cv, scoring="f1", random_state=42
)
svc_random_search.fit(X_train, y_train)

# Best parameter combination & F1 score:
print(f"SVC Random Search Best Params: {svc_random_search.best_params_}")  # best params
print(f"SVC Random Search Best CV F1: {svc_random_search.best_score_:.3f}")  # best F1

best_C_svc = svc_random_search.best_params_["classifier__C"]
best_kernel_svc = svc_random_search.best_params_["classifier__kernel"]
best_gamma_svc = svc_random_search.best_params_["classifier__gamma"]

SVC Random Search Best Params: {'classifier__kernel': 'linear', 'classifier__gamma': 'scale', 'classifier__C': 10}
SVC Random Search Best CV F1: 0.974


In [12]:
# Fine Tuning via GridSearch: Dependent on random search space
svc_grid_param = {
    "classifier__C": [
        best_C_svc * 0.1,
        best_C_svc * 0.5,
        best_C_svc,  # based on random search
        best_C_svc * 2,
        best_C_svc * 10,
    ],
    "classifier__kernel": [best_kernel_svc],
    "classifier__gamma": [best_gamma_svc],
}

# Perform search and fit on training data:
svc_grid_search = GridSearchCV(svc_pipeline, svc_grid_param, cv=cv, scoring="f1")
svc_grid_search.fit(X_train, y_train)

# Best parameter combination & F1 score:
print(f"SVC Fine-Tuned Grid Search Best Params: {svc_grid_search.best_params_}")
print(f"SVC Fine-Tuned Search Best CV F1: {svc_grid_search.best_score_:.3f}")  # best F1

SVC Fine-Tuned Grid Search Best Params: {'classifier__C': 20, 'classifier__gamma': 'scale', 'classifier__kernel': 'linear'}
SVC Fine-Tuned Search Best CV F1: 0.978


In [13]:
# Coarse Tuning via RandomSearch:
# many of these will be sampled randomly
rf_random_param = {
    "classifier__n_estimators": randint(50, 500),
    "classifier__max_depth": [None, 5, 10, 15, 20, 30, 50],
    "classifier__min_samples_split": randint(5, 20),
}

# Perform search and fit on training data:
rf_random_search = RandomizedSearchCV(
    rf_pipeline, rf_random_param, cv=cv, scoring="f1", random_state=42
)
rf_random_search.fit(X_train, y_train)

# Best parameter combination & F1 score:
print(f"RF Random Search Best Params: {rf_random_search.best_params_}")  # best params
print(f"RF Random Search Best CV F1: {rf_random_search.best_score_:.3f}")  # best F1

best_n_rf = rf_random_search.best_params_["classifier__n_estimators"]
best_depth_rf = rf_random_search.best_params_["classifier__max_depth"]
best_minsamples_rf = rf_random_search.best_params_["classifier__min_samples_split"]

RF Random Search Best Params: {'classifier__max_depth': 20, 'classifier__min_samples_split': 8, 'classifier__n_estimators': 409}
RF Random Search Best CV F1: 0.941


In [14]:
# Fine Tuning via GridSearch: Dependent on random search space
rf_grid_param = {
    "classifier__n_estimators": [best_n_rf - 100, best_n_rf, best_n_rf + 100],
    "classifier__max_depth": [
        best_depth_rf - 5,
        best_depth_rf - 4,
        best_depth_rf - 3,
        best_depth_rf,
    ],
    "classifier__min_samples_split": [
        max(2, best_minsamples_rf - 2),
        best_minsamples_rf,
        best_minsamples_rf + 2,
    ],
}

# Perform search and fit on training data:
rf_grid_search = GridSearchCV(rf_pipeline, rf_grid_param, cv=cv, scoring="f1")
rf_grid_search.fit(X_train, y_train)

# Best parameter combination & F1 score:
print(f"RF Fine-Tuned Grid Search Best Params: {rf_grid_search.best_params_}")
print(f"RF Fine-Tuned Search Best CV F1: {rf_grid_search.best_score_:.3f}")  # best F1

RF Fine-Tuned Grid Search Best Params: {'classifier__max_depth': 15, 'classifier__min_samples_split': 6, 'classifier__n_estimators': 309}
RF Fine-Tuned Search Best CV F1: 0.946


In [ ]:
# Coarse Tuning via RandomSearch:
xgb_random_param = {
    "classifier__n_estimators": randint(50, 500),
    "classifier__learning_rate": [0.001, 0.01, 0.1, 0.3, 0.5],
    "classifier__max_depth": [None, 5, 10, 15, 20, 30, 50],
}

# Perform search and fit on training data:
xgb_random_search = RandomizedSearchCV(
    xgb_pipeline, xgb_random_param, cv=cv, scoring="f1", random_state=42
)
xgb_random_search.fit(X_train, y_train)

# Best parameter combination & F1 score:
print(f"XGB Random Search Best Params: {xgb_random_search.best_params_}")  # best params
print(f"XGB Random Search Best CV F1: {xgb_random_search.best_score_:.3f}")  # best F1

best_lrate_xgb = xgb_random_search.best_params_["classifier__learning_rate"]
best_n_xgb = xgb_random_search.best_params_["classifier__n_estimators"]
best_depth_xgb = xgb_random_search.best_params_["classifier__max_depth"]

XGB Random Search Best Params: {'classifier__learning_rate': 0.5, 'classifier__max_depth': 15, 'classifier__n_estimators': 409}
XGB Random Search Best CV F1: 0.938


In [16]:
# Fine Tuning via GridSearch: Dependent on random search space
xgb_grid_param = {
    "classifier__learning_rate": [
        best_lrate_xgb * 0.5,
        best_lrate_xgb,
        best_lrate_xgb * 0.8,
    ],
    "classifier__n_estimators": [best_n_xgb - 100, best_n_xgb, best_n_xgb + 100],
    "classifier__max_depth": [
        best_depth_xgb - 5,
        best_depth_xgb - 4,
        best_depth_xgb - 3,
        best_depth_xgb,
    ],
}

# Perform search and fit on training data:
xgb_grid_search = GridSearchCV(xgb_pipeline, xgb_grid_param, cv=cv, scoring="f1")
xgb_grid_search.fit(X_train, y_train)

# Best parameter combination & F1 score:
print(f"XGB Fine-Tuned Grid Search Best Params: {xgb_grid_search.best_params_}")
print(f"XGB Fine-Tuned Search Best CV F1: {xgb_grid_search.best_score_:.3f}")  # best F1

XGB Fine-Tuned Grid Search Best Params: {'classifier__learning_rate': 0.4, 'classifier__max_depth': 10, 'classifier__n_estimators': 309}
XGB Fine-Tuned Search Best CV F1: 0.947


##### Comparing ML Models with Best Hyperparameters

F1 score and ROC-AUC will continue to be used, in order to determine whether each model can correctly distinguish between the biotic and abiotic class. 

In [17]:
# Dictionary gathering best hyperparameters across all four models:
models = {
    "Logistic Regression": lr_grid_search.best_estimator_,
    "SVC": svc_grid_search.best_estimator_,
    "Random Forest": rf_grid_search.best_estimator_,
    "XGBoost": xgb_grid_search.best_estimator_,
}

results = {}  # dict for results
for c_name, model in models.items():
    y_pred = model.predict(X_test)  # predictions on testing set
    y_prob = model.predict_proba(X_test)[:, 1]  # probability of predictions
    f1 = f1_score(y_test, y_pred)  # F1 score
    auc = roc_auc_score(y_test, y_prob)  # AUC ROC score
    # Across 5 CV folds, evaluate score:
    cv_f1 = {
        "Logistic Regression": lr_grid_search.best_score_,
        "SVC": svc_grid_search.best_score_,
        "Random Forest": rf_grid_search.best_score_,
        "XGBoost": xgb_grid_search.best_score_,
    }[c_name]
    results[c_name] = {"cv_f1": cv_f1, "f1": f1, "auc": auc}
    # Output results:
    print(f"\n{c_name} Results:")
    print(f"CV F1: {cv_f1:.3f} | F1: {f1:.3f} | AUC ROC: {auc:.3f}")


Logistic Regression Results:
CV F1: 0.973 | F1: 0.931 | AUC ROC: 0.702

SVC Results:
CV F1: 0.978 | F1: 0.966 | AUC ROC: 0.702

Random Forest Results:
CV F1: 0.946 | F1: 0.947 | AUC ROC: 0.976

XGBoost Results:
CV F1: 0.947 | F1: 0.982 | AUC ROC: 0.988


##### Conclusions:

Extending the analysis from the previous [notebook](../notebooks/03_balanced_ml_classification.ipynb) exploring classification performance on a reduced class-imbalanced subset, classifier performance on the entire biotic-abiotic dataset of $154$ samples improved across all four classifiers (LR, SVC, RF, XGBoost). This improvement is more prominent and noticeable across the two tree-based models (RF and XGBoost), which achieved high test-set F1 scores ($0.947, 0.982$) and AUC-ROC scores ($0.976, 0.988$). These results imply that the models were better able to capture chemical differences to discriminate between biotic and abiotic samples, rather than overfitting on highly-imbalanced training data. 

Although LR and SVC achieved high CV F1 scores, they yielded far lower AUC-ROC scores ($0.702, 0.702$) on the test set, implying that linear and distance-based classifiers struggled with predictions under the severe class imbalance between biotics and abiotics ($10:1$). Furthermore, convergence warnings were observed during training for both LR and SVC, indicating that the optimiser did not fully converge within `max_iter=500`. This required significantly raising the value of this hyperparameter to $2000$ in order to ensure optimiser stability.

Consequently, XGBoost is the strongest performer with a test-set F1 score of $0.982$ and AUC-ROC score of $0.988$, followed by RF. These final results heavily suggest that ensemble tree-based classifiers were best suited to capturing the chemical diversity of the py-GC-MS sample set while managing predictions under a high class imbalance. Based on the frameworks developed across this and the [preceding notebook](../notebooks/03_balanced_ml_classification.ipynb), the [`classifiers.py`](../mlbiosig/classifiers.py) module within `mlbiosig` was created to perform automated imbalance handling, `Pipeline` construction, coarse-to-fine hyperparameter tuning, and classifier training prior to model evaluation.